# LesioScan — Notebook 02: Image Preprocessing & DataLoader
**Tujuan:** menjadikan hasil Notebook 01 siap dipakai model ShuffleNet V2 tanpa menyalin gambar. **Notebook ini tidak melatih model.**

**Alur:** Drive + clone GitHub → verifikasi `split.csv` → transform train/eval → smoke test tensor → visualisasi → simpan konfigurasi dan audit di Drive.

**Penting:** protokol splitting saat ini **70/15/15 patient-level**, bukan 5-fold CV. Jangan gunakan test set untuk tuning.

## 1. Mount Google Drive dan ambil source code terbaru

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import subprocess,sys,os
REPO=Path('/content/lesioscan-ml')
if not (REPO/'.git').is_dir():
    subprocess.run(['git','clone','https://github.com/Liex2810/lesioscan-ml.git',str(REPO)],check=True)
else:
    subprocess.run(['git','pull','--ff-only','origin','main'],cwd=REPO,check=True)
for name in ['scripts/preprocess.py','src/lesioscan/dataset.py','src/lesioscan/preprocessing.py','src/lesioscan/preprocessing_pipeline.py']:
    assert (REPO/name).is_file(), f'Belum ada di GitHub: {name}'
print('✅ Source code siap:',REPO)

## 2. Set path dan konfigurasi

In [ ]:
DRIVE_ROOT=Path('/content/drive/MyDrive/LesioScan-AI')
DATASET=DRIVE_ROOT/'datasets/PAD-UFES-20'
SPLIT=DRIVE_ROOT/'experiments/splits/split.csv'
PREPROCESS_OUT=DRIVE_ROOT/'experiments/preprocessing'
PREPROCESS_OUT.mkdir(parents=True,exist_ok=True)
assert DATASET.is_dir(),f'Dataset tidak ada: {DATASET}'
assert SPLIT.is_file(),f'Split tidak ada: {SPLIT}'
IMAGE_SIZE=224
BATCH_SIZE=16
NUM_WORKERS=0  # Aman untuk akses Google Drive di Colab
FULL_SCAN=False  # True hanya jika ingin membaca ulang semua gambar
print('✅ Dataset asli dipakai langsung; tidak ada duplikasi gambar')

## 3. Unit test modul (data sintetis, tidak membaca PAD-UFES-20)

In [ ]:
subprocess.run([sys.executable,'-m','pytest','tests/test_preprocessing.py','-q'],cwd=REPO,check=True)

## 4. Jalankan preprocessing audit — dengan progress status

In [ ]:
import re,time,threading
import ipywidgets as widgets
from IPython.display import display
bar=widgets.IntProgress(value=0,min=0,max=100,description='Prep:',bar_style='info',layout=widgets.Layout(width='700px'))
status=widgets.HTML('Status: Memulai...');clock=widgets.HTML('Waktu: 0 detik')
display(bar,status,clock)
cmd=[sys.executable,'-u','scripts/preprocess.py','--split',str(SPLIT),'--images',str(DATASET),'--out',str(PREPROCESS_OUT),'--image-size',str(IMAGE_SIZE),'--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
if FULL_SCAN:cmd.append('--full-scan')
start=time.monotonic(); stop=threading.Event()
def timer():
    while not stop.wait(1):
        seconds=int(time.monotonic()-start)
        clock.value=f'Waktu: {seconds//60} menit {seconds%60} detik'
thread=threading.Thread(target=timer,daemon=True);thread.start()
process=subprocess.Popen(cmd,cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
lines=[]
try:
    for line in process.stdout:
        lines.append(line.rstrip())
        mark=re.search(r'\[PREPROCESS\s+(\d+)%\]\s*(.*)',line)
        if mark:
            bar.value=int(mark.group(1));status.value=f'Status: {mark.group(2)}'
        elif line.strip() and not line.lstrip().startswith('{'):
            # Simpan log lengkap untuk debug tanpa memenuhi tampilan notebook.
            pass
    rc=process.wait()
    if rc:
        bar.bar_style='danger';status.value='❌ Preprocessing gagal'
        print('\n'.join(lines[-40:]))
        raise RuntimeError(f'Error preprocess.py: {rc}')
    bar.value=100;bar.bar_style='success';status.value='✅ Preprocessing selesai'
finally:
    stop.set();thread.join(timeout=2);process.stdout.close()

## 5. Tampilkan grafik, tabel, dan konfigurasi yang tersimpan

In [ ]:
import pandas as pd,json
from IPython.display import Image as DisplayImage, display
with open(PREPROCESS_OUT/'preprocessing_summary.json',encoding='utf-8') as f:
    summary=json.load(f)
print(json.dumps(summary,indent=2))
display(pd.read_csv(PREPROCESS_OUT/'tables/split_class_distribution.csv'))
display(pd.read_csv(PREPROCESS_OUT/'tables/label_mapping_and_weights.csv'))
display(DisplayImage(filename=str(PREPROCESS_OUT/'figures/preprocessing_preview.png'),width=900))

## 6. Pengujian langsung Dataset/DataLoader untuk Notebook 03

In [ ]:
sys.path.insert(0,str(REPO/'src'))
from lesioscan.dataset import load_and_validate_split,make_dataloaders,training_class_weights
from lesioscan.preprocessing import build_transforms,CLASSES
frame,image_paths,distribution=load_and_validate_split(SPLIT,DATASET)
train_tf,eval_tf=build_transforms(IMAGE_SIZE)
loaders,datasets=make_dataloaders(frame,image_paths,train_tf,eval_tf,batch_size=BATCH_SIZE,num_workers=NUM_WORKERS)
images,labels=next(iter(loaders['train']))
print('Batch tensor:',tuple(images.shape),'| Labels:',labels.tolist())
assert images.shape[1:]==(3,IMAGE_SIZE,IMAGE_SIZE)
print('✅ Loader train/val/test siap dipakai oleh scripts/train.py di Notebook 03')

## 7. Hasil dan keputusan arsitektur
- **Tidak menyimpan salinan gambar atau tensor seluruh dataset.** Transformasi diterapkan saat DataLoader membaca gambar.
- **Training** menggunakan augmentasi ringan; **validation/test** hanya transformasi deterministik.
- Statistik dan class weights dihitung hanya dari **training set**, agar test tidak memengaruhi pelatihan.
- Output tetap di `experiments/preprocessing/` pada Google Drive.
- Notebook selanjutnya: **03_model_training.ipynb** (feature extraction, fine-tuning, checkpoint), menggunakan modul `preprocessing.py` dan `dataset.py` yang sama.
- Tidak untuk diagnosis medis.